# Smartphone Addiction Data Analysis

Let's explore the addiction prediction dataset and build a model

In [14]:
import pandas as pd

df = pd.read_csv("../data/data.csv")

print("First few records:")
print(df.head())
print("\nData structure:")
print(df.info())
print("\nBasic statistics:")
print(df.describe())

First few records:
  transaction_id user_id  age gender  daily_screen_time_hours  \
0       TXN00001  U00001   21   Male                     3.23   
1       TXN00002  U00002   24  Other                     5.09   
2       TXN00003  U00003   31  Other                     6.06   
3       TXN00004  U00004   32  Other                     7.83   
4       TXN00005  U00005   25   Male                     9.96   

   social_media_hours  gaming_hours  work_study_hours  sleep_hours  \
0                2.01          0.89              4.55         7.55   
1                3.81          2.24              4.44         7.66   
2                1.36          3.83              2.35         4.92   
3                5.85          1.51              3.54         8.23   
4                5.92          3.42              5.27         6.21   

   notifications_per_day  app_opens_per_day  weekend_screen_time stress_level  \
0                    248                154                 3.95       Medium   
1      

## Clean Up the Data

Remove columns we don't need and encode category values

In [15]:
df.drop(['transaction_id', 'user_id', 'addiction_level'], axis=1, inplace=True)

In [16]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['gender', 'stress_level', 'academic_work_impact']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [17]:
print("Data ready!")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

Data ready!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      7500 non-null   int64  
 1   gender                   7500 non-null   int64  
 2   daily_screen_time_hours  7500 non-null   float64
 3   social_media_hours       7500 non-null   float64
 4   gaming_hours             7500 non-null   float64
 5   work_study_hours         7500 non-null   float64
 6   sleep_hours              7500 non-null   float64
 7   notifications_per_day    7500 non-null   int64  
 8   app_opens_per_day        7500 non-null   int64  
 9   weekend_screen_time      7500 non-null   float64
 10  stress_level             7500 non-null   int64  
 11  academic_work_impact     7500 non-null   int64  
 12  addicted_label           7500 non-null   int64  
dtypes: float64(6), int64(7)
memory usage: 761.8 KB
None

Missing value

## Prepare Features and Target

Split the data into what we use to predict and what we're predicting

In [18]:
X = df.drop('addicted_label', axis=1)
y = df['addicted_label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (7500, 12)
Target shape: (7500,)


## Train and Test Split

Split 80% for training and 20% for testing

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("Features scaled!")

Training set: 6000 samples
Test set: 1500 samples
Features scaled!


## Train the Model

Build and train a prediction model

In [20]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


## Evaluate the Model

Check how well it works on the test data

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("Performance Results:")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Not Addicted', 'Addicted']))

Performance Results:
Accuracy:  0.9347 (93.47%)
Precision: 0.9608
Recall:    0.9463
F1-Score:  0.9535
ROC-AUC:   0.9893
              precision    recall  f1-score   support

Not Addicted       0.87      0.91      0.89       438
    Addicted       0.96      0.95      0.95      1062

    accuracy                           0.93      1500
   macro avg       0.92      0.93      0.92      1500
weighted avg       0.94      0.93      0.94      1500



## Detailed Results

In [22]:
cm = confusion_matrix(y_test, y_pred)

print("\nResults Breakdown:")
print("=" * 50)
print(f"Correctly found not addicted: {cm[0, 0]}")
print(f"Wrongly marked as addicted: {cm[0, 1]}")
print(f"Wrongly marked as not addicted: {cm[1, 0]}")
print(f"Correctly found addicted: {cm[1, 1]}")


Results Breakdown:
Correctly found not addicted: 397
Wrongly marked as addicted: 41
Wrongly marked as not addicted: 57
Correctly found addicted: 1005


## Save the Model

Export the trained model for use in predictions

In [23]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(model, "../models/addiction_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(X.columns.tolist(), "../models/features.pkl")

print("✅ Model saved!")
print("   Brain: ../models/addiction_model.pkl")
print("   Scaler: ../models/scaler.pkl")
print("   Features: ../models/features.pkl")

✅ Model saved!
   Brain: ../models/addiction_model.pkl
   Scaler: ../models/scaler.pkl
   Features: ../models/features.pkl
